# AWARE Dog Emotion — Separate Four-Class Image Model

**Training is deliberately disabled.** The Kaggle record declares CC0, but the image-source and rights chain is unresolved. Do not enable training or export weights until the uploader confirms image rights for this use, labels are reviewed, and the user explicitly approves training.

This notebook uses the audited image archive with angry, happy, relaxed, and sad folders. It does not fine-tune, replace, or validate the original five-class AniMind checkpoint. These labels are weakly sourced annotations, not verified emotional states. No results are prefilled.


## 1. Runtime and dependencies
Choose Runtime → Change runtime type → GPU if available. The notebook checks the actual accelerator, does not assume a T4, keeps mixed precision off, and uses small batches.


In [ ]:
import os, sys, json, hashlib, zipfile, subprocess, shutil
from pathlib import Path
import tensorflow as tf
print("Python:", sys.version, "TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc: print("Memory growth unchanged:", exc)
print("Mixed precision disabled")


In [ ]:
%pip -q install imagehash scikit-learn pandas matplotlib seaborn pillow opencv-python-headless


## 2. Private dataset import and closed training gates
Download the Kaggle archive yourself and store it in private Drive. Audit cells may read it; training and model export remain disabled by default.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ARCHIVE_PATH = Path("/content/drive/MyDrive/AWARE/private/dog-emotion.zip")
EXTRACT_ROOT = Path("/content/aware_private_dataset")
OUTPUT_DIR = Path("/content/drive/MyDrive/AWARE/private/four_class_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RIGHTS_CONFIRMED = False
LABELS_VERIFIED = False
TRAINING_APPROVED = False
ENABLE_TRAINING = False
EXPORT_APPROVED = False
CLASS_NAMES = ["angry", "happy", "relaxed", "sad"]
IMG_SIZE = 224
print("Archive exists:", ARCHIVE_PATH.exists(), "training enabled:", ENABLE_TRAINING)


## 3. ZIP integrity, safe extraction, and image inventory
Validate the archive before extracting. If the archive structure differs, inspect and update DATA_ROOT rather than guessing.


In [ ]:
import pandas as pd
from PIL import Image
from collections import defaultdict
if not ARCHIVE_PATH.exists(): raise FileNotFoundError("Place the approved private archive at ARCHIVE_PATH.")
with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    bad = zf.testzip()
    if bad: raise ValueError(f"Corrupt ZIP member: {bad}")
    root = EXTRACT_ROOT.resolve()
    for info in zf.infolist():
        if not (EXTRACT_ROOT / info.filename).resolve().is_relative_to(root):
            raise ValueError(f"Unsafe archive path: {info.filename}")
    if not EXTRACT_ROOT.exists(): zf.extractall(EXTRACT_ROOT)
print("Archive bytes:", ARCHIVE_PATH.stat().st_size)
print("Archive SHA-256:", hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest())
print("Top-level entries:", [p.name for p in EXTRACT_ROOT.iterdir()][:20])
data_roots = [p for p in EXTRACT_ROOT.rglob("*") if p.is_dir() and {c.name.lower() for c in p.iterdir() if c.is_dir()} >= set(CLASS_NAMES)]
if not data_roots: raise FileNotFoundError("Four class folders not found; inspect the layout above.")
DATA_ROOT = data_roots[0]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
paths = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
rows=[]; bad_images=[]
for p in paths:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im:
            w,h=im.size
            im.load()
        rows.append({"path":str(p.relative_to(DATA_ROOT)),"label_folder":p.parent.name.lower(),
                     "bytes":p.stat().st_size,"width":w,"height":h,"sha256":sha256_file(p)})
    except Exception as exc: bad_images.append({"path":str(p),"error":str(exc)})
manifest=pd.DataFrame(rows)
print("Decoded:",len(manifest),"failures:",len(bad_images),"labels:",sorted(manifest.label_folder.unique()))
display(manifest.head())
if bad_images: display(pd.DataFrame(bad_images).head(20))


## 4. Annotation review and class distribution
Inspect any labels.csv and reconcile it with folder labels. The dataset description says labels were manually assigned, but the annotation protocol and image provenance need human review. Counts alone do not verify labels.


In [ ]:
annotation_files=list(EXTRACT_ROOT.rglob("labels.csv"))
for p in annotation_files:
    labels_csv=pd.read_csv(p)
    print("Annotation:",p,"rows:",len(labels_csv),"columns:",labels_csv.columns.tolist())
    display(labels_csv.head(10))
import matplotlib.pyplot as plt
counts=manifest.label_folder.value_counts().reindex(CLASS_NAMES,fill_value=0)
display(counts.rename("image_count").to_frame())
counts.plot.bar(title="Images per annotated class"); plt.ylabel("Images"); plt.show()
print("Unexpected:",sorted(set(manifest.label_folder)-set(CLASS_NAMES)))
print("Dimensions:"); display(manifest[["width","height","bytes"]].describe())
assert set(manifest.label_folder)==set(CLASS_NAMES),"Review class folders before continuing."


## 5. Exact and perceptual duplicate detection
Exact SHA-256 identifies byte-identical images. Perceptual hashes group near-identical images to prevent split leakage. Review cross-label matches because they may reflect conflicting annotations.


In [ ]:
import imagehash
from itertools import combinations
manifest["phash"]=[str(imagehash.phash(Image.open(DATA_ROOT/rel).convert("RGB"))) for rel in manifest.path]
print("Files in exact-duplicate groups:",int(manifest.sha256.duplicated(keep=False).sum()))
hashes=[int(x,16) for x in manifest.phash]
buckets=defaultdict(list)
for i,value in enumerate(hashes):
    for part in range(5):
        width=13 if part<4 else 12
        shift=64-(part+1)*13 if part<4 else 0
        buckets[(part,(value>>shift)&((1<<width)-1))].append(i)
candidates=set()
for members in buckets.values():
    for a,b in combinations(members,2): candidates.add((min(a,b),max(a,b)))
parent=list(range(len(manifest)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[rb]=ra
pairs=[]
for a,b in candidates:
    distance=(hashes[a]^hashes[b]).bit_count()
    if distance<=4:
        union(a,b)
        pairs.append({"path_a":manifest.iloc[a].path,"label_a":manifest.iloc[a].label_folder,
                      "path_b":manifest.iloc[b].path,"label_b":manifest.iloc[b].label_folder,"hamming":distance})
manifest["duplicate_group"]=[find(i) for i in range(len(manifest))]
near=pd.DataFrame(pairs)
print("Perceptual pairs with Hamming distance <= 4:",len(near))
if len(near):
    display(near.head(30))
    print("Cross-label pairs:",int((near.label_a!=near.label_b).sum()))
manifest.to_csv(OUTPUT_DIR/"dataset_manifest_private.csv",index=False)
near.to_csv(OUTPUT_DIR/"perceptual_duplicates_private.csv",index=False)


## 6. Visual review and grouped stratified split
Inspect example images and duplicate matches. Split by duplicate component so similar images cannot cross train, validation, and test sets.


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(len(CLASS_NAMES),4,figsize=(12,3*len(CLASS_NAMES)))
for r,label in enumerate(CLASS_NAMES):
    for c,(_,row) in enumerate(manifest[manifest.label_folder==label].head(4).iterrows()):
        axes[r,c].imshow(Image.open(DATA_ROOT/row.path).convert("RGB"))
        axes[r,c].set_title(label); axes[r,c].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
y=manifest.label_folder.to_numpy()
groups=manifest.duplicate_group.to_numpy()
idx=np.arange(len(manifest))
# Approximate 70/15/15 while grouping perceptual duplicates.
outer=list(StratifiedGroupKFold(n_splits=20,shuffle=True,random_state=42).split(idx,y,groups))
test_idx=np.concatenate([outer[i][1] for i in range(3)])
trainval_idx=np.setdiff1d(idx,test_idx)
inner=list(StratifiedGroupKFold(n_splits=17,shuffle=True,random_state=43).split(trainval_idx,y[trainval_idx],groups[trainval_idx]))
val_idx=np.concatenate([inner[i][1] for i in range(3)])
train_idx=np.setdiff1d(trainval_idx,val_idx)
train_df=manifest.iloc[train_idx].reset_index(drop=True)
val_df=manifest.iloc[val_idx].reset_index(drop=True)
test_df=manifest.iloc[test_idx].reset_index(drop=True)
assert set(train_df.duplicate_group).isdisjoint(val_df.duplicate_group)
assert set(train_df.duplicate_group).isdisjoint(test_df.duplicate_group)
assert set(val_df.duplicate_group).isdisjoint(test_df.duplicate_group)
for name,split in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(name,len(split),split.label_folder.value_counts().reindex(CLASS_NAMES,fill_value=0).to_dict())
    assert set(split.label_folder)==set(CLASS_NAMES),f"{name} split misses a class."


## 7. Separate transfer-learning model and input contract
Use ImageNet MobileNetV2 features with a new four-class head. The old five-class checkpoint is not loaded or changed. This new artifact accepts RGB float32 images in [0,1], resized to 224×224; its model graph rescales to [-1,1].


In [ ]:
BATCH_SIZE=16
AUTOTUNE=tf.data.AUTOTUNE
def decode_image(path,label_id):
    raw=tf.io.read_file(path)
    image=tf.io.decode_image(raw,channels=3,expand_animations=False)
    image.set_shape([None,None,3])
    image=tf.image.resize(image,[IMG_SIZE,IMG_SIZE],antialias=True)
    return tf.cast(image,tf.float32)/255.0,label_id
class_to_id={name:i for i,name in enumerate(CLASS_NAMES)}
def make_dataset(split,training=False):
    filenames=[str(DATA_ROOT/rel) for rel in split.path]
    labels=[class_to_id[label] for label in split.label_folder]
    ds=tf.data.Dataset.from_tensor_slices((filenames,labels))
    if training: ds=ds.shuffle(min(len(filenames),1000),seed=42,reshuffle_each_iteration=True)
    return ds.map(decode_image,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
train_ds=make_dataset(train_df,True)
val_ds=make_dataset(val_df)
test_ds=make_dataset(test_df)
print("Input batch:",next(iter(train_ds))[0].shape)


## 8. Closed training gate
This cell intentionally stops before training. Only proceed after rights confirmation, label review, and explicit user approval.


In [ ]:
if not (RIGHTS_CONFIRMED and LABELS_VERIFIED and TRAINING_APPROVED and ENABLE_TRAINING):
    raise RuntimeError("Training disabled: confirm rights and labels and obtain explicit approval before enabling all four flags.")


## 9. Training with augmentation and early stopping
After approval, train a frozen pretrained feature extractor with small batches, class weights, early stopping, and learning-rate reduction. Mixed precision remains disabled.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
augment=tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
],name="train_augmentation")
base=tf.keras.applications.MobileNetV2(include_top=False,weights="imagenet",input_shape=(IMG_SIZE,IMG_SIZE,3))
base.trainable=False
inputs=tf.keras.Input((IMG_SIZE,IMG_SIZE,3),name="rgb_float_0_1")
x=augment(inputs)
x=tf.keras.layers.Rescaling(2.0,offset=-1.0,name="mobilenetv2_rescale")(x)
x=base(x,training=False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dropout(0.25)(x)
outputs=tf.keras.layers.Dense(len(CLASS_NAMES),activation="softmax",name="four_class_head")(x)
model=tf.keras.Model(inputs,outputs,name="aware_dog_emotion_four_class")
y_train=np.array([class_to_id[label] for label in train_df.label_folder])
cw=compute_class_weight(class_weight="balanced",classes=np.arange(len(CLASS_NAMES)),y=y_train)
class_weights={i:float(v) for i,v in enumerate(cw)}
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.25,patience=2,min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(OUTPUT_DIR/"best_four_class.keras",monitor="val_loss",save_best_only=True),
]
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
history=model.fit(train_ds,validation_data=val_ds,epochs=25,batch_size=BATCH_SIZE,class_weight=class_weights,callbacks=callbacks)


## 10. Training curves and untouched test evaluation
Evaluate once after selecting the checkpoint with validation data. Report class-wise precision, recall, F1, confusion matrix, and actual sample counts. These are dataset estimates, not scientific validation of affect.


In [ ]:
from sklearn.metrics import classification_report,confusion_matrix
import seaborn as sns
best_path=OUTPUT_DIR/"best_four_class.keras"
model=tf.keras.models.load_model(best_path,compile=False)
y_true=np.concatenate([labels.numpy() for _,labels in test_ds])
probabilities=model.predict(test_ds,verbose=0)
y_pred=probabilities.argmax(axis=1)
print(classification_report(y_true,y_pred,labels=list(range(len(CLASS_NAMES))),target_names=CLASS_NAMES,zero_division=0))
cm=confusion_matrix(y_true,y_pred,labels=list(range(len(CLASS_NAMES))))
sns.heatmap(cm,annot=True,fmt="d",xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Observed"); plt.title("Untouched test-set confusion matrix"); plt.show()
history_frame=pd.DataFrame(history.history)
history_frame[["loss","val_loss"]].plot(title="Loss"); plt.show()
history_frame[["accuracy","val_accuracy"]].plot(title="Accuracy"); plt.show()
history_frame.to_csv(OUTPUT_DIR/"training_history.csv",index=False)


## 11. AWARE preprocessing compatibility check and private export
The check uses the same OpenCV resize and float scaling as the current AWARE preprocessor, then converts BGR to RGB for this separate model. It validates input/output shapes and normalized scores. It does not activate this model in the Windows application; a separate reviewed adapter is required.


In [ ]:
import cv2
def aware_four_class_input(image_path):
    bgr=cv2.imread(str(image_path))
    if bgr is None: raise ValueError(f"Could not decode image: {image_path}")
    resized=cv2.resize(bgr,(IMG_SIZE,IMG_SIZE))
    rgb=cv2.cvtColor(resized,cv2.COLOR_BGR2RGB)
    return np.expand_dims(rgb.astype(np.float32)/255.0,axis=0)
sample=aware_four_class_input(DATA_ROOT/test_df.iloc[0].path)
probe=tf.keras.models.load_model(best_path,compile=False)
scores=probe.predict(sample,verbose=0)[0]
assert sample.shape==(1,IMG_SIZE,IMG_SIZE,3) and scores.shape==(len(CLASS_NAMES),)
assert np.isfinite(scores).all() and np.isclose(scores.sum(),1.0,atol=1e-4)
print("Input/output contract PASS:",sample.shape,scores.shape,float(scores.sum()))
if not EXPORT_APPROVED:
    print("Export skipped; review rights and held-out results first.")
else:
    export_dir=Path("/content/aware_model")
    export_dir.mkdir(parents=True,exist_ok=True)
    probe.save(export_dir/"model.keras")
    (export_dir/"class_mapping.json").write_text(json.dumps({str(i):n for i,n in enumerate(CLASS_NAMES)},indent=2))
    (export_dir/"preprocessing.json").write_text(json.dumps({
        "input_shape":[224,224,3],"input_color_order":"RGB","input_dtype":"float32",
        "input_range":[0.0,1.0],"resize":"OpenCV cv2.resize to 224x224",
        "normalization":"divide by 255; model rescales to [-1,1]","backend_adapter_required":True
    },indent=2))
    (export_dir/"model_metadata.json").write_text(json.dumps({
        "model_name":"AWARE separate four-class image model",
        "architecture":"MobileNetV2 ImageNet feature extractor plus new four-class head",
        "classes":CLASS_NAMES,"test_split_images":int(len(test_df)),
        "weights_publication":"not approved; keep private"
    },indent=2))
    archive=shutil.make_archive("/content/aware_model","zip",export_dir)
    from google.colab import files
    files.download(archive)


## 12. Colab steps and alternative data plan

1. Open this notebook in Google Colab, select a GPU if available, and mount Drive.
2. Put the downloaded archive in the private Drive location shown above.
3. Run integrity, label, duplicate, visualization, and split cells for audit only. Training is disabled.
4. If rights provenance remains unclear, do not train. Prefer consented or directly licensed images with item-level creator/source records, documented class definitions, and expert-reviewed labels.
5. After rights confirmation, label review, and explicit user approval, set the four training flags to True and rerun the notebook from the top. Keep EXPORT_APPROVED False until the held-out report is reviewed.
6. Keep dataset files and weights private. Never copy the artifact over backend/model/dog_emotion_best_model.keras. The four-class model is separate; Windows integration requires a dedicated adapter driven by the exported mapping and preprocessing metadata.
